# Semantic

```
pip install -U spacy
# python -m spacy download en_core_web_lg
```

In [ ]:
# import spacy.cli
# spacy.cli.download("en_core_web_md")
# en_core_web_md
# en_core_web_lg 

# Phrases, Verbs

In [1]:
url = "https://jamming-bot.arthew0.online/api/storage_latest/"
import requests

# Получаем данные с указанного URL
response = requests.get(url)
if response.status_code == 200:
    json_data = response.json()
    # Основные данные находятся в ключе "data"
    parsed_data = json_data.get("data", [])

else:
    print(f"Ошибка при получении данных: {response.status_code}")

In [ ]:
import csv
from datetime import datetime

# Задаём имя файла для CSV
csv_filename = "dataset_table.csv"
fieldnames = [
    'number', 'url', 'src', 'ip', 'status_code', 'timestamp', 'city',
    'latitude', 'longitude', 'tags', 'words', 'hrases', 'entities',
    'text_length', 'semantic', 'semantic_words', 'semantic_hrases', 
    'screenshot_url', 'text'
]

def extract_field(entry, key, default=""):
    return entry.get(key, default) if key in entry else default

rows = []
for idx, entry in enumerate(parsed_data, 1):
    # Собираем значения полей, по вашим требованиям
    number = idx
    url = extract_field(entry, "url")
    src = extract_field(entry, "src")
    ip = extract_field(entry, "ip")
    status_code = extract_field(entry, "status_code")
    # Обеспечиваем строку времени или пустую строку
    timestamp_val = extract_field(entry, "timestamp")
    try:
        # Преобразуем если нужно к читаемому виду
        if isinstance(timestamp_val, (float, int)):
            timestamp = datetime.fromtimestamp(timestamp_val).isoformat()
        elif isinstance(timestamp_val, str):
            timestamp = timestamp_val
        else:
            timestamp = ""
    except Exception:
        timestamp = str(timestamp_val)
    city = extract_field(entry, "city")
    latitude = extract_field(entry, "latitude")
    longitude = extract_field(entry, "longitude")
    tags = ",".join(entry.get("tags", [])) if isinstance(entry.get("tags", []), list) else entry.get("tags", "")
    words = ",".join(entry.get("words", [])) if isinstance(entry.get("words", []), list) else entry.get("words", "")
    hrases = ",".join(entry.get("hrases", [])) if isinstance(entry.get("hrases", []), list) else entry.get("hrases", "")
    entities = ",".join(entry.get("entities", [])) if isinstance(entry.get("entities", []), list) else entry.get("entities", "")
    text_length = len(entry.get("text", "")) if entry.get("text", None) else 0
    semantic = extract_field(entry, "semantic")
    semantic_words = ",".join(entry.get("semantic_words", [])) if isinstance(entry.get("semantic_words", []), list) else entry.get("semantic_words", "")
    semantic_hrases = ",".join(entry.get("semantic_hrases", [])) if isinstance(entry.get("semantic_hrases", []), list) else entry.get("semantic_hrases", "")
    screenshot_url = extract_field(entry, "screenshot_url")
    text = extract_field(entry, "text")

    row = [
        number, url, src, ip, status_code, timestamp, city, latitude, longitude,
        tags, words, hrases, entities, text_length, semantic, semantic_words, semantic_hrases,
        screenshot_url, text
    ]
    rows.append(row)

# Запись в CSV
with open(csv_filename, 'w', encoding='utf-8', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(fieldnames)
    writer.writerows(rows)

print(f"CSV-таблица успешно сформирована: {csv_filename}")
for entry in parsed_data:
    for key, value in entry.items():
        print(f"{key}: {value}")

In [4]:
import pandas as pd
csv_filename = "dataset_table.csv"
df = pd.read_csv(csv_filename, on_bad_lines='skip')
df.head()

,number,url,src,ip,status_code,timestamp,city,latitude,longitude,tags,words,hrases,entities,text_length,semantic,semantic_words,semantic_hrases,screenshot_url,text
0,1,https://www.971thefan.com/event/matty-ice-at-m...,https://www.971thefan.com/event/matty-ice-at-m...,32.192.146.176,200,1773773304.507513,Ashburn,39.0438,-77.4874,[],[],[],[],0,NaN,NaN,NaN,NaN,NaN
1,2,https://www.971thefan.com/venue/mcdaniel-gmc/,https://www.971thefan.com/event/matty-ice-at-m...,32.192.146.176,200,1773773310.183659,Ashburn,39.0438,-77.4874,[],[],[],[],0,NaN,NaN,NaN,NaN,NaN
2,3,https://www.971thefan.com/brandon-beam/,https://www.971thefan.com/show/the-buckeye-show/,32.192.146.176,200,1773773318.060816,Ashburn,39.0438,-77.4874,[],[],[],[],0,NaN,NaN,NaN,NaN,NaN
3,4,https://www.971thefan.com/podcast/buckeye-show...,https://www.971thefan.com/show/the-buckeye-show/,100.49.187.32,200,1773773323.716649,Ashburn,39.0438,-77.4874,[],[],[],[],0,NaN,NaN,NaN,https://s3.firstvds.ru/steps/screenshots/00270...,NaN
4,5,https://www.971thefan.com/bobby-carpenter/,https://www.971thefan.com/show/morning-juice/,100.49.187.32,200,1773773329.817404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN


In [6]:
import re  
import string


#
# PREPARE
#
def remove_punctuation(text):
    return re.sub(r'[:;!?“”]', '', text)

def clean_corpus(corpus):
    # Remove punctuations from the corpus
    translator = str.maketrans('', '', string.punctuation)
    corpus = corpus.translate(translator)
    # Remove digits from the corpus
    remove_digits = str.maketrans('', '', string.digits)
    corpus = corpus.translate(remove_digits)
    return corpus

def prepare_text(text):
    if not isinstance(text, str) or not text:
        text = ""
    text = remove_punctuation(text)
    text = clean_corpus(text)
    return text

#
# CLEAN
#
def remove_all_cjk(text):
    return re.sub(r'[\u3000-\u30FF\u4E00-\u9FFF\uAC00-\uD7AF]', '', text)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    if text==None or not text or text=="nan":
        text = "-"
    text = text.strip().replace("\n","").replace("\r","").replace("\t"," ").strip()
    text = re.sub(r'\s+', ' ', text.strip())
    text = remove_all_cjk(text)
    return text

In [7]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

# Установите punkt, если еще не установлено
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')

def is_sentence_nltk(text):
    sentences = sent_tokenize(text)
    if len(sentences) != 1 or text.strip() != sentences[0]:
        return False
    words = word_tokenize(text)
    words = [word for word in words if word.isalnum()]
    if len(words) < 2:
        return False
    return True

def get_first_two_words(text):
    text = ' '.join(text.split())
    words = word_tokenize(text)
    filtered_words = [word for word in words if word.isalnum()]
    if len(filtered_words) < 3:
        return ""
    return filtered_words[:2]

# Примеры
# print(is_sentence_nltk("I would like to receive the version of Newsletter"))  # True
# print(is_sentence_nltk("！！"))
# print(is_sentence_nltk("Это предложение. А это другое."))  # False
# print(is_sentence_nltk("Mr. Smith lives here."))  # True

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\arthew0\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\arthew0\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\arthew0\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [11]:

csv_analyzed_filename = "dataset_analyzed.csv"
with open(csv_analyzed_filename, 'w', encoding='utf-8', newline='') as csvanalyzedfile:
    writer = csv.writer(csvanalyzedfile)
    writer.writerow(["number", "title", "prepated_text", "cleaned_text"])
    for _, row in df.iterrows():
        text = row['text']
        cleaned_text = clean_text(text)
        prepated_text = prepare_text(cleaned_text)
        if is_sentence_nltk(cleaned_text):
            title = " ".join(get_first_two_words(text))            
            writer.writerow([row['number'], title, prepated_text, cleaned_text])
            # print(f"{row['number']} : {title} : {prepated_text} : {cleaned_text}")


In [14]:
#import pandas as pd
dataset_analyzed = "dataset_analyzed.csv"
df = pd.read_csv(dataset_analyzed, on_bad_lines='skip')
df.head()

for index, row in df.iterrows():
    print(row['cleaned_text'])


Please check your DOPPLER 10 FORECAST at WBNS-10TV to prepare yourself for Race Day.
We use cookies and data to
Packet Pick Up
Stephanie Breedlove
You’re an entrepreneur, which inevitably means you’ve got a mess of ideas for your business, whether they’re piling up in your notes app, scribbled amongst the pages of a trusty notebook, or floating around aimlessly in your head.
MADDY & MEGANANNA & GRACETIMIROSE & URIMICHELLE & BRYAN
GET YOUR FREE BOOKKEEPING PLAYBOOK
GET YOUR FREE BOOKKEEPING PLAYBOOK
Stephanie Breedlove
GET YOUR FREE BOOKKEEPING PLAYBOOK
GET YOUR FREE BOOKKEEPING PLAYBOOK
GET YOUR FREE BOOKKEEPING PLAYBOOK
Box Office Information:
You’re an entrepreneur, which inevitably means you’ve got a mess of ideas for your business, whether they’re piling up in your notes app, scribbled amongst the pages of a trusty notebook, or floating around aimlessly in your head.
Cake Flavors
Let us create a dessert display for your wedding or special event!
Let us create a dessert display for 